# 11. Train observability predictor

This notebook trains the new opt-in observability-prediction model. The target for each physical calibration is the seven-dimensional dimensionless soft-projector coordinate observability, ordered as

`[phi_x, phi_y, phi_z, rho_x, rho_y, rho_z, tau]`.

Workflow:
1. load KAIST windows;
2. compute and cache raw-window observability targets in RAM;
3. apply stochastic input augmentation;
4. geometry-process and tokenize measurements;
5. run global Transformer attention and post-attention per-calibration pooling;
6. predict one bounded 7D observability vector per calibration;
7. train with componentwise MSE;
8. run a small fixed-batch verification, full training, plots, and held-out test metrics.

The target cache is computed before stochastic augmentation. Calibration-event augmentation is therefore disabled here. Sampling-rate augmentation is enabled as an explicit first experiment, but it changes information content directly and is the first ablation to try if target consistency is poor.


## 1. Imports and repository root


In [ ]:
from pathlib import Path
import inspect
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "obscalib").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the nn-obs-calib repository root containing src/obscalib.")

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations.config import AugmentationConfig, CalibrationEventConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, SE3NoiseConfig, VectorNoiseConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.calibration.observability_targets import ObservabilityTargetCache
from obscalib.config import ObservabilityPredictionHeadConfig, ObservabilityPredictionModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import MeasurementType
from obscalib.geometry.processing import GeometryProcessor
from obscalib.losses.observability_prediction_loss import ObservabilityPredictionLoss
from obscalib.models.observability_prediction_model import ObservabilityPredictionModel
from obscalib.observability.estimators import ObservabilityMatrixEstimator
from obscalib.observability.layout import CalibrationParameterLayout
from obscalib.observability.mappings import dimensionless_soft_projector_observability
from obscalib.observability.numerics import ObservabilityNumericsConfig
from obscalib.observability.structures import ObservabilityResult
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint
from obscalib.training.metrics_logger import MetricsLogger

print("Repository root:", ROOT)
print("PyTorch:", torch.__version__)


## 2. Experiment parameters


In [ ]:
RUN_NAME = "kaist_observability_predictor_001"

RANDOM_SEED = 12345
TRAIN_AUGMENTATION_SEED = 23456
VALIDATION_AUGMENTATION_SEED = 34567
TEST_AUGMENTATION_SEED = 45678
SMALL_TEST_AUGMENTATION_SEED = 56789

DEVICE_NAME = "cuda"
DEVICE = torch.device("cuda" if DEVICE_NAME == "auto" and torch.cuda.is_available() else "cpu" if DEVICE_NAME == "auto" else DEVICE_NAME)
MODEL_DTYPE = torch.float32

WINDOW_DURATION_S = 5.0
WINDOW_STRIDE_S = 5.0
MAX_SAMPLES_PER_SENSOR = 700

CALIBRATION_KEYS = ("imu", "lidar")
CALIBRATION_COMPONENT_NAMES = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z", "tau")

BATCH_SIZE = 16
NUM_WORKERS = 0
DROP_LAST_TRAIN = True

MEASUREMENT_DIM = 6
TOKEN_INPUT_DIM = MEASUREMENT_DIM + 2
D_MODEL = 128
N_HEADS = 8
NUM_TRANSFORMER_LAYERS = 12
DIM_FEEDFORWARD = 512
TRANSFORMER_DROPOUT = 0.1
NUM_SUMMARY_TOKENS = 1
HEAD_HIDDEN_DIMS = (128, 64)
HEAD_DROPOUT = 0.1
HEAD_MODE = "individual"  # "individual" or "shared"

DIMENSIONLESS_ROTATION_SCALE_RAD = np.deg2rad(5.0)
DIMENSIONLESS_TRANSLATION_SCALE_M = 0.05
DIMENSIONLESS_TIME_OFFSET_SCALE_S = 0.015
DIMENSIONLESS_CALIBRATION_SCALE = (DIMENSIONLESS_ROTATION_SCALE_RAD, DIMENSIONLESS_ROTATION_SCALE_RAD, DIMENSIONLESS_ROTATION_SCALE_RAD, DIMENSIONLESS_TRANSLATION_SCALE_M, DIMENSIONLESS_TRANSLATION_SCALE_M, DIMENSIONLESS_TRANSLATION_SCALE_M, DIMENSIONLESS_TIME_OFFSET_SCALE_S)
LAMBDA_VALUE_SETTING = 0.25
USE_NJIT_OBSERVABILITY = True
PROJECTION_NUMERICAL_TOLERANCE = 1e-10
REFERENCE_LIDAR_STREAM_KEY = "lidar"

NUM_EPOCHS = 100
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE_EPOCHS = 5
CHECKPOINT_EVERY_EPOCHS = 10
PRINT_EVERY_BATCHES = 200

RUN_SMALL_VERIFICATION = True
SMALL_TEST_BATCH_SIZE = 16
SMALL_TEST_STEPS = 100
SMALL_TEST_LEARNING_RATE = 1e-3

RESUME_CHECKPOINT = None
# RESUME_CHECKPOINT = ROOT / "runs" / RUN_NAME / "last.pt"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Device:", DEVICE)
print("Head mode:", HEAD_MODE)
print("Token input dimension:", TOKEN_INPUT_DIM)


## 3. Dataset paths, split, calibration, and windows


In [ ]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")

TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES

if len(ALL_SEQUENCE_NAMES) != len(set(ALL_SEQUENCE_NAMES)):
    raise ValueError("Train, validation, and test sequence sets overlap.")

T_B_L = np.array([
    [-0.514066, -0.702201, -0.492595, -0.440699],
    [0.486485, -0.711672, 0.506809, 0.397052],
    [-0.706447, 0.0208933, 0.707457, 1.90953],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

T_I_B = np.array([
    [1.0, 0.0, 0.0, -0.07],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.7],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

T_B_I = np.linalg.inv(T_I_B)
rig_calibration = KAISTRigCalibration(T_B_I=T_B_I, T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)

print("train:", TRAIN_NAMES)
print("validation:", VAL_NAMES)
print("test:", TEST_NAMES)


## 4. Load datasets and DataLoaders


In [ ]:
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT)
spec_by_name = {spec.name: spec for spec in all_specs}

train_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TRAIN_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
val_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in VAL_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
test_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TEST_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)

print(train_dataset.summary()); print(); print(val_dataset.summary()); print(); print(test_dataset.summary())

train_shuffle_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=DROP_LAST_TRAIN, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)
test_loader = make_kaist_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("test batches:", len(test_loader))


## 5. Scientific observability estimator


In [ ]:
calibration_layout = CalibrationParameterLayout.from_calibration_keys(CALIBRATION_KEYS)

GYRO_MEASUREMENT_COVARIANCE = torch.tensor([[2.96367932e-06, -1.94438078e-07, -4.83643437e-07], [-1.94438078e-07, 1.89162212e-06, 1.09062693e-07], [-4.83643437e-07, 1.09062693e-07, 1.88843506e-06]], dtype=torch.float64)
ACCEL_MEASUREMENT_COVARIANCE = torch.tensor([[4.85003041e-03, -9.25262862e-05, 2.14219676e-04], [-9.25262862e-05, 1.51980006e-03, -1.13182999e-03], [2.14219676e-04, -1.13182999e-03, 2.29777687e-03]], dtype=torch.float64)
LIDAR_SCAN_TO_MAP_ROTATION_COVARIANCE = torch.tensor([[1.10442767e-05, 9.46809587e-07, 3.78536515e-06], [9.46809587e-07, 3.35041997e-05, 1.90008728e-06], [3.78536515e-06, 1.90008728e-06, 1.10553483e-05]], dtype=torch.float64)
LIDAR_SCAN_TO_MAP_TRANSLATION_COVARIANCE = torch.tensor([[0.00843466, 0.00266052, 0.00702046], [0.00266052, 0.01444850, 0.00225705], [0.00702046, 0.00225705, 0.00704268]], dtype=torch.float64)
LIDAR_SCAN_TO_MAP_COVARIANCE = torch.zeros((6, 6), dtype=torch.float64)
LIDAR_SCAN_TO_MAP_COVARIANCE[:3, :3] = LIDAR_SCAN_TO_MAP_ROTATION_COVARIANCE
LIDAR_SCAN_TO_MAP_COVARIANCE[3:, 3:] = LIDAR_SCAN_TO_MAP_TRANSLATION_COVARIANCE
LIDAR_RELATIVE_RESIDUAL_COVARIANCE = 2.0 * LIDAR_SCAN_TO_MAP_COVARIANCE

residual_covariance_by_stream = {"gyro": GYRO_MEASUREMENT_COVARIANCE, "accel": ACCEL_MEASUREMENT_COVARIANCE, "lidar": LIDAR_RELATIVE_RESIDUAL_COVARIANCE}
GRAVITY_WORLD = torch.tensor([0.0, 0.0, 9.81], dtype=torch.float64)
gyro_bias_by_calibration_key = {"imu": torch.zeros(3, dtype=torch.float64)}
observability_numerics = ObservabilityNumericsConfig(relative_tolerance=PROJECTION_NUMERICAL_TOLERANCE, parameter_scales=None)


def relative_lidar_body_trajectory_provider(measurements, metadata, calibration, reference_timebase):
    stream_key = reference_timebase.stream_key
    if stream_key != REFERENCE_LIDAR_STREAM_KEY:
        raise ValueError(f"Expected reference stream {REFERENCE_LIDAR_STREAM_KEY!r}, got {stream_key!r}.")
    if stream_key not in measurements or stream_key not in metadata:
        raise KeyError(f"Missing reference LiDAR stream {stream_key!r}.")

    stream = measurements[stream_key]
    if stream.interval_start_timestamps is None:
        raise ValueError("Reference LiDAR stream must carry interval_start_timestamps.")

    calibration_key = metadata[stream_key].calibration_key
    T_B_L_current = calibration[calibration_key].transform
    if T_B_L_current.ndim == 3:
        if T_B_L_current.shape[0] != 1:
            raise ValueError("Unbatched trajectory provider expects one calibration state.")
        T_B_L_current = T_B_L_current[0]

    T_B_L_current = T_B_L_current.to(device="cpu", dtype=torch.float64)
    T_L_B_current = torch.linalg.inv(T_B_L_current)
    relative_lidar_poses = stream.values
    interval_starts = stream.interval_start_timestamps
    interval_ends = stream.timestamps

    if relative_lidar_poses.ndim == 4:
        if relative_lidar_poses.shape[0] != 1:
            raise ValueError("Trajectory provider expects an unbatched stream or singleton batch.")
        relative_lidar_poses = relative_lidar_poses[0]
        interval_starts = interval_starts[0]
        interval_ends = interval_ends[0]

    relative_lidar_poses = relative_lidar_poses.to(device="cpu", dtype=torch.float64)
    interval_starts = interval_starts.to(device="cpu", dtype=torch.float64)
    interval_ends = interval_ends.to(device="cpu", dtype=torch.float64)
    reference_timestamps = reference_timebase.timestamps.to(device="cpu", dtype=torch.float64)

    poses_by_time = {}
    current_pose = torch.eye(4, dtype=torch.float64)
    for interval_index in range(relative_lidar_poses.shape[0]):
        start_time = float(interval_starts[interval_index].item())
        end_time = float(interval_ends[interval_index].item())
        poses_by_time[start_time] = current_pose
        current_pose = current_pose @ (T_B_L_current @ relative_lidar_poses[interval_index] @ T_L_B_current)
        poses_by_time[end_time] = current_pose

    available_times = np.asarray(sorted(poses_by_time), dtype=float)
    trajectory_poses = []
    for timestamp in reference_timestamps:
        query_time = float(timestamp.item())
        nearest_index = int(np.argmin(np.abs(available_times - query_time)))
        nearest_time = float(available_times[nearest_index])
        if abs(nearest_time - query_time) > 1e-6:
            raise ValueError(f"Could not construct nominal pose at reference timestamp {query_time:.9f}; nearest available time is {nearest_time:.9f}.")
        trajectory_poses.append(poses_by_time[nearest_time])

    return torch.stack(trajectory_poses, dim=0)


estimator_signature = inspect.signature(ObservabilityMatrixEstimator.__init__)
estimator_kwargs = {"calibration_layout": calibration_layout, "trajectory_provider": relative_lidar_body_trajectory_provider, "residual_covariance_by_stream": residual_covariance_by_stream}
optional_estimator_kwargs = {"gravity_world": GRAVITY_WORLD, "gyro_bias_by_calibration_key": gyro_bias_by_calibration_key, "numerics": observability_numerics, "njit": USE_NJIT_OBSERVABILITY, "reference_stream_key": REFERENCE_LIDAR_STREAM_KEY}
for argument_name, argument_value in optional_estimator_kwargs.items():
    if argument_name in estimator_signature.parameters:
        estimator_kwargs[argument_name] = argument_value

missing_required_arguments = [name for name, parameter in estimator_signature.parameters.items() if name != "self" and parameter.kind in {inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY} and parameter.default is inspect.Parameter.empty and name not in estimator_kwargs]
if missing_required_arguments:
    raise RuntimeError(f"ObservabilityMatrixEstimator has required constructor arguments not configured here: {missing_required_arguments}. Signature: {estimator_signature}")

observability_estimator = ObservabilityMatrixEstimator(**estimator_kwargs)
print("Calibration dimension:", calibration_layout.total_dimension)
print("Estimator signature:", estimator_signature)


## 6. Build RAM target caches


In [ ]:
def call_observability_estimator(window):
    forward_signature = inspect.signature(observability_estimator.forward)
    if "metadata" in forward_signature.parameters:
        return observability_estimator(window.streams, window.current_calibration, window.metadata)
    return observability_estimator(window.streams, window.current_calibration)


def build_target_cache(dataset, *, description: str) -> ObservabilityTargetCache:
    values = torch.empty((len(dataset), len(CALIBRATION_KEYS), 7), dtype=torch.float32, device="cpu")
    observability_estimator.eval()

    with torch.no_grad():
        for dataset_index in tqdm(range(len(dataset)), desc=description):
            window = collate_windows([dataset[dataset_index]])
            try:
                result = call_observability_estimator(window)
                if not isinstance(result, ObservabilityResult):
                    raise TypeError("Observability estimator must return ObservabilityResult.")
                if result.raw is None or not hasattr(result.raw, "fisher_information_matrix"):
                    raise TypeError("ObservabilityResult.raw must expose fisher_information_matrix.")

                fisher = result.raw.fisher_information_matrix
                if fisher.ndim == 3:
                    if fisher.shape[0] != 1:
                        raise ValueError("Singleton cache construction expected one Fisher matrix.")
                    fisher = fisher[0]

                mapped = dimensionless_soft_projector_observability(fisher, calibration_layout, calibration_scale=DIMENSIONLESS_CALIBRATION_SCALE, lambda_value_setting=LAMBDA_VALUE_SETTING)
                coordinate_observability = mapped[0].detach().to(device="cpu", dtype=torch.float32)

                for calibration_index, calibration_key in enumerate(CALIBRATION_KEYS):
                    values[dataset_index, calibration_index] = coordinate_observability[calibration_layout.block_for(calibration_key).parameter_slice]
            except Exception as error:
                sequence_name = dataset.sequence_name_for_window(dataset_index) if hasattr(dataset, "sequence_name_for_window") else "<unknown>"
                raise RuntimeError(f"Observability target construction failed for dataset index {dataset_index}, sequence {sequence_name!r}.") from error

    return ObservabilityTargetCache(values=values, calibration_keys=CALIBRATION_KEYS)


target_cache_start = time.perf_counter()
train_target_cache = build_target_cache(train_dataset, description="train observability targets")
val_target_cache = build_target_cache(val_dataset, description="validation observability targets")
test_target_cache = build_target_cache(test_dataset, description="test observability targets")
print(f"Target caches built in {time.perf_counter() - target_cache_start:.1f} s")
print("train cache:", tuple(train_target_cache.values.shape))
print("validation cache:", tuple(val_target_cache.values.shape))
print("test cache:", tuple(test_target_cache.values.shape))


## 7. Inspect target distribution


In [ ]:
rows = []
for calibration_index, calibration_key in enumerate(CALIBRATION_KEYS):
    for component_index, component_name in enumerate(CALIBRATION_COMPONENT_NAMES):
        x = train_target_cache.values[:, calibration_index, component_index].numpy()
        rows.append({"calibration": calibration_key, "component": component_name, "min": float(x.min()), "mean": float(x.mean()), "std": float(x.std()), "max": float(x.max())})
display(pd.DataFrame(rows))


## 8. Training augmentation


In [ ]:
USE_AUGMENTATIONS = True
USE_SAMPLING_RATE_AUGMENTATION = True

if USE_AUGMENTATIONS:
    augmentation_config = AugmentationConfig(
        sampling_rate=SamplingRateAugmentationConfig(enabled=USE_SAMPLING_RATE_AUGMENTATION, minimum_imu_frequency_hz=40.0, minimum_lidar_frequency_hz=5.0, imu_probability=0.8, lidar_probability=0.5),
        frame_randomization=FrameRandomizationConfig(enabled=True),
        prior_perturbation=PriorPerturbationConfig(enabled=True, probability=0.8, rotation=PerturbationMagnitudeConfig(maximum=np.deg2rad(15.0)), translation=PerturbationMagnitudeConfig(maximum=0.15), time_offset=PerturbationMagnitudeConfig(maximum=0.03)),
        calibration_event=CalibrationEventConfig(enabled=False),
        noise=NoiseAugmentationConfig(enabled=True, vector_by_type={MeasurementType.IMU_GYROSCOPE: VectorNoiseConfig(gaussian_std=0.01, window_bias_std=0.003), MeasurementType.IMU_ACCELEROMETER: VectorNoiseConfig(gaussian_std=0.08, window_bias_std=0.03)}, se3_by_type={MeasurementType.LIDAR_POSE: SE3NoiseConfig(rotation_std=np.deg2rad(0.15), translation_std=0.01)}),
    )
else:
    augmentation_config = AugmentationConfig()

augmentation_pipeline = AugmentationPipeline(augmentation_config)
print("Use augmentations:", USE_AUGMENTATIONS)
print("Use sampling-rate augmentation:", USE_SAMPLING_RATE_AUGMENTATION)


## 9. Model and forward path


In [ ]:
model_config = ObservabilityPredictionModelConfig(
    transformer=TransformerConfig(input_dim=TOKEN_INPUT_DIM, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_TRANSFORMER_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=TRANSFORMER_DROPOUT, activation="gelu", num_summary_tokens=NUM_SUMMARY_TOKENS),
    calibration_keys=CALIBRATION_KEYS,
    head=ObservabilityPredictionHeadConfig(hidden_dims=HEAD_HIDDEN_DIMS, activation="gelu", dropout=HEAD_DROPOUT),
    head_mode=HEAD_MODE,
)

geometry_processor = GeometryProcessor()
tokenizer = Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=CALIBRATION_KEYS)
loss_function = ObservabilityPredictionLoss()


def build_model():
    return ObservabilityPredictionModel(model_config).to(device=DEVICE, dtype=MODEL_DTYPE)


def forward_observability_model(model, window):
    canonical_streams = geometry_processor(window.streams, window.metadata, window.current_calibration)
    tokens = tokenizer(canonical_streams, observability=None)
    if tokens.calibration_ids is None:
        raise ValueError("Observability-prediction Tokenizer must emit calibration_ids.")
    return model(tokens), tokens


def make_augmentation_generator(seed: int):
    return torch.Generator(device=DEVICE.type).manual_seed(seed)


model = build_model()
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 10. One-batch sanity check


In [ ]:
san_base = collate_windows([train_dataset[i] for i in range(min(BATCH_SIZE, len(train_dataset)))])
if san_base.dataset_indices is None:
    raise ValueError("WindowBatch.dataset_indices is required for target lookup.")

san_targets = train_target_cache.targets_for_indices(san_base.dataset_indices, device=DEVICE, dtype=MODEL_DTYPE)
san_window = window_batch_to(san_base, DEVICE, dtype=MODEL_DTYPE)
if USE_AUGMENTATIONS:
    san_window = augmentation_pipeline(san_window, generator=make_augmentation_generator(SMALL_TEST_AUGMENTATION_SEED)).augmented_window

model.eval()
with torch.no_grad():
    san_output, san_tokens = forward_observability_model(model, san_window)
    san_loss = loss_function(san_output, san_targets).total

print("Token tensor:", tuple(san_tokens.x.shape))
print("Calibration IDs:", tuple(san_tokens.calibration_ids.shape))
print("Sanity total loss:", float(san_loss.item()))
for key in CALIBRATION_KEYS:
    print(key, "prediction", tuple(san_output.predictions[key].shape), "target", tuple(san_targets[key].shape), "prediction range", (float(san_output.predictions[key].min()), float(san_output.predictions[key].max())))


## 11. Small fixed-batch overfit verification


In [ ]:
if RUN_SMALL_VERIFICATION:
    torch.manual_seed(RANDOM_SEED + 1)
    verification_model = build_model()
    verification_optimizer = torch.optim.AdamW(verification_model.parameters(), lr=SMALL_TEST_LEARNING_RATE, weight_decay=0.0)

    base = collate_windows([train_dataset[i] for i in range(min(SMALL_TEST_BATCH_SIZE, len(train_dataset)))])
    targets = train_target_cache.targets_for_indices(base.dataset_indices, device=DEVICE, dtype=MODEL_DTYPE)
    fixed_window = window_batch_to(base, DEVICE, dtype=MODEL_DTYPE)
    if USE_AUGMENTATIONS:
        fixed_window = augmentation_pipeline(fixed_window, generator=make_augmentation_generator(SMALL_TEST_AUGMENTATION_SEED)).augmented_window

    verification_model.eval()
    with torch.no_grad():
        initial_loss = float(loss_function(forward_observability_model(verification_model, fixed_window)[0], targets).total.item())

    verification_history = []
    verification_model.train()
    for step in range(SMALL_TEST_STEPS):
        verification_optimizer.zero_grad(set_to_none=True)
        output, _ = forward_observability_model(verification_model, fixed_window)
        loss = loss_function(output, targets).total
        if not torch.isfinite(loss):
            raise ValueError(f"Non-finite small-verification loss at step {step}.")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(verification_model.parameters(), 1.0)
        verification_optimizer.step()
        verification_history.append(float(loss.detach().item()))

    verification_model.eval()
    with torch.no_grad():
        final_loss = float(loss_function(forward_observability_model(verification_model, fixed_window)[0], targets).total.item())

    print("Small verification initial loss:", initial_loss)
    print("Small verification final loss:", final_loss)
    print("Final / initial:", final_loss / initial_loss)
    if final_loss >= initial_loss:
        raise RuntimeError("Small fixed-batch verification did not reduce the loss.")

    plt.figure(figsize=(10, 4)); plt.plot(np.arange(1, SMALL_TEST_STEPS + 1), verification_history); plt.xlabel("optimization step"); plt.ylabel("MSE loss"); plt.title("Small fixed-batch observability overfit test"); plt.grid(True); plt.show()
    del verification_model, verification_optimizer
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


## 12. Fresh full-training model, optimizer, scheduler


In [ ]:
torch.manual_seed(RANDOM_SEED)
model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_REDUCTION_FACTOR, patience=LR_PATIENCE_EPOCHS)


## 13. Epoch metrics and train/evaluation function


In [ ]:
class MetricAccumulator:
    def __init__(self):
        self.loss_sum = 0.0
        self.num_samples = 0
        self.abs_sum = {key: torch.zeros(7, dtype=torch.float64) for key in CALIBRATION_KEYS}
        self.sq_sum = {key: torch.zeros(7, dtype=torch.float64) for key in CALIBRATION_KEYS}

    def update(self, loss, output, targets, batch_size):
        self.loss_sum += float(loss.detach().item()) * batch_size
        self.num_samples += batch_size
        for key in CALIBRATION_KEYS:
            error = (output.predictions[key].detach() - targets[key].detach()).cpu().double()
            self.abs_sum[key] += torch.abs(error).sum(dim=0)
            self.sq_sum[key] += error.square().sum(dim=0)

    def finalize(self):
        metrics = {"loss_total": self.loss_sum / self.num_samples}
        for key in CALIBRATION_KEYS:
            mae = self.abs_sum[key] / self.num_samples
            mse = self.sq_sum[key] / self.num_samples
            metrics[f"{key}/mae"] = float(mae.mean())
            metrics[f"{key}/mse"] = float(mse.mean())
            metrics[f"{key}/rmse"] = float(torch.sqrt(mse.mean()))
            for i, name in enumerate(CALIBRATION_COMPONENT_NAMES):
                metrics[f"{key}/{name}_mae"] = float(mae[i])
                metrics[f"{key}/{name}_rmse"] = float(torch.sqrt(mse[i]))
        return metrics


def run_epoch(loader, target_cache, *, training: bool, augmentation_generator, global_step: int):
    model.train() if training else model.eval()
    accumulator = MetricAccumulator()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    with (torch.enable_grad() if training else torch.no_grad()):
        for batch_index, base_window in enumerate(loader):
            if base_window.dataset_indices is None:
                raise ValueError("WindowBatch.dataset_indices is required for cache lookup.")

            targets = target_cache.targets_for_indices(base_window.dataset_indices, device=DEVICE, dtype=MODEL_DTYPE)
            working_window = window_batch_to(base_window, DEVICE, dtype=MODEL_DTYPE)
            if USE_AUGMENTATIONS:
                working_window = augmentation_pipeline(working_window, generator=augmentation_generator).augmented_window

            if training:
                optimizer.zero_grad(set_to_none=True)

            output, _ = forward_observability_model(model, working_window)
            loss = loss_function(output, targets).total
            if not torch.isfinite(loss):
                raise ValueError(f"Non-finite loss in batch {batch_index}.")

            if training:
                loss.backward()
                if GRADIENT_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                optimizer.step()
                global_step += 1

            batch_size = next(iter(working_window.streams.values())).values.shape[0]
            accumulator.update(loss, output, targets, batch_size)
            if training and PRINT_EVERY_BATCHES > 0 and (batch_index + 1) % PRINT_EVERY_BATCHES == 0:
                print(f"batch {batch_index + 1:5d}/{len(loader):5d} loss={float(loss.detach()):.6f}")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    metrics = accumulator.finalize()
    metrics["epoch_time_s"] = elapsed
    metrics["samples_per_s"] = accumulator.num_samples / elapsed
    return metrics, global_step


## 14. Run directory, checkpoint state, and resume


In [ ]:
RUN_DIR = ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / "metrics.jsonl"
LAST_CHECKPOINT_PATH = RUN_DIR / "last.pt"
BEST_CHECKPOINT_PATH = RUN_DIR / "best.pt"

EXPERIMENT_SNAPSHOT = {
    "schema_version": 1,
    "model_config": model_config,
    "measurement_dim": MEASUREMENT_DIM,
    "calibration_keys": CALIBRATION_KEYS,
    "windowing_config": windowing_config,
    "rig_calibration": rig_calibration,
    "augmentation_config": augmentation_config,
    "observability_target_config": {"rotation_scale_rad": DIMENSIONLESS_ROTATION_SCALE_RAD, "translation_scale_m": DIMENSIONLESS_TRANSLATION_SCALE_M, "time_offset_scale_s": DIMENSIONLESS_TIME_OFFSET_SCALE_S, "lambda_value_setting": LAMBDA_VALUE_SETTING, "use_njit": USE_NJIT_OBSERVABILITY, "reference_lidar_stream_key": REFERENCE_LIDAR_STREAM_KEY},
    "sequence_split": {"train": tuple(TRAIN_NAMES), "validation": tuple(VAL_NAMES), "test": tuple(TEST_NAMES)},
    "training_config": {"run_name": RUN_NAME, "model_dtype": str(MODEL_DTYPE), "random_seed": RANDOM_SEED, "head_mode": HEAD_MODE},
}

train_augmentation_generator = make_augmentation_generator(TRAIN_AUGMENTATION_SEED)
start_epoch = 0
global_step = 0
best_validation_loss = float("inf")

if RESUME_CHECKPOINT is not None:
    checkpoint = load_training_checkpoint(RESUME_CHECKPOINT, model=model, optimizer=optimizer, scheduler=scheduler, generators={"train_augmentation": train_augmentation_generator, "train_shuffle": train_shuffle_generator}, map_location=DEVICE)
    start_epoch = int(checkpoint.get("epoch", 0))
    global_step = int(checkpoint.get("global_step", 0))
    best_validation_loss = float(checkpoint.get("extra_state", {}).get("best_validation_loss", float("inf")))
    print("Resumed from:", RESUME_CHECKPOINT)
    print("start epoch:", start_epoch)
    print("global step:", global_step)

print("Run directory:", RUN_DIR)


## 15. Full training


In [ ]:
history = []

with MetricsLogger(METRICS_PATH, append=RESUME_CHECKPOINT is not None) as logger:
    if RESUME_CHECKPOINT is None:
        logger.log_config(model_config, name="model_config")
        logger.log_config(augmentation_config, name="augmentation_config")
        logger.log_config(EXPERIMENT_SNAPSHOT["observability_target_config"], name="observability_target_config")
        logger.log_config({"device": str(DEVICE), "batch_size": BATCH_SIZE, "window_duration_s": WINDOW_DURATION_S, "window_stride_s": WINDOW_STRIDE_S, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "train_sequences": TRAIN_NAMES, "validation_sequences": VAL_NAMES, "test_sequences": TEST_NAMES}, name="experiment")

    for epoch in range(start_epoch, NUM_EPOCHS):
        print(); print("=" * 80); print(f"Epoch {epoch + 1}/{NUM_EPOCHS}"); print("=" * 80)

        train_metrics, global_step = run_epoch(train_loader, train_target_cache, training=True, augmentation_generator=train_augmentation_generator, global_step=global_step)
        val_metrics, _ = run_epoch(val_loader, val_target_cache, training=False, augmentation_generator=make_augmentation_generator(VALIDATION_AUGMENTATION_SEED), global_step=global_step)

        scheduler.step(val_metrics["loss_total"])
        lr = optimizer.param_groups[0]["lr"]
        print(f"train loss={train_metrics['loss_total']:.6f} val loss={val_metrics['loss_total']:.6f} imu val MAE={val_metrics['imu/mae']:.4f} lidar val MAE={val_metrics['lidar/mae']:.4f} lr={lr:.3e}")

        logger.log_metrics({**train_metrics, "learning_rate": lr}, split="train", epoch=epoch + 1, step=global_step)
        logger.log_metrics(val_metrics, split="validation", epoch=epoch + 1, step=global_step)
        history.append({"epoch": epoch + 1, "train_loss": train_metrics["loss_total"], "val_loss": val_metrics["loss_total"], "train_imu_mae": train_metrics["imu/mae"], "val_imu_mae": val_metrics["imu/mae"], "train_lidar_mae": train_metrics["lidar/mae"], "val_lidar_mae": val_metrics["lidar/mae"], "learning_rate": lr})

        if val_metrics["loss_total"] < best_validation_loss:
            best_validation_loss = val_metrics["loss_total"]
            save_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch + 1, global_step=global_step, metrics=val_metrics, generators={"train_augmentation": train_augmentation_generator, "train_shuffle": train_shuffle_generator}, extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT})
            print("Saved new best checkpoint.")

        if (epoch + 1) % CHECKPOINT_EVERY_EPOCHS == 0 or epoch + 1 == NUM_EPOCHS:
            save_training_checkpoint(LAST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch + 1, global_step=global_step, metrics=val_metrics, generators={"train_augmentation": train_augmentation_generator, "train_shuffle": train_shuffle_generator}, extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT})
            print("Saved last checkpoint.")


## 16. Plot training history


In [ ]:
if history:
    epochs = [row["epoch"] for row in history]
    plt.figure(figsize=(12, 5)); plt.plot(epochs, [row["train_loss"] for row in history], label="train"); plt.plot(epochs, [row["val_loss"] for row in history], label="validation"); plt.xlabel("epoch"); plt.ylabel("MSE loss"); plt.title("Observability-prediction training loss"); plt.grid(True); plt.legend(); plt.show()
    plt.figure(figsize=(12, 5)); plt.plot(epochs, [row["train_imu_mae"] for row in history], label="train IMU"); plt.plot(epochs, [row["val_imu_mae"] for row in history], label="validation IMU"); plt.plot(epochs, [row["train_lidar_mae"] for row in history], label="train LiDAR"); plt.plot(epochs, [row["val_lidar_mae"] for row in history], label="validation LiDAR"); plt.xlabel("epoch"); plt.ylabel("mean absolute observability error"); plt.title("Per-calibration observability MAE"); plt.grid(True); plt.legend(); plt.show()
    plt.figure(figsize=(12, 4)); plt.plot(epochs, [row["learning_rate"] for row in history]); plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.title("Learning-rate schedule"); plt.grid(True); plt.show()
else:
    print("No new epochs were run in this notebook session.")


## 17. Evaluate best checkpoint and report test metrics


In [ ]:
load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
test_metrics, _ = run_epoch(test_loader, test_target_cache, training=False, augmentation_generator=make_augmentation_generator(TEST_AUGMENTATION_SEED), global_step=global_step)

print("Test metrics:")
for key, value in sorted(test_metrics.items()):
    print(f"{key}: {value}")

report_rows = []
for calibration_key in CALIBRATION_KEYS:
    for component_name in CALIBRATION_COMPONENT_NAMES:
        report_rows.append({"calibration": calibration_key, "component": component_name, "MAE": test_metrics[f"{calibration_key}/{component_name}_mae"], "RMSE": test_metrics[f"{calibration_key}/{component_name}_rmse"]})

test_report = pd.DataFrame(report_rows)
print(); print("Best-checkpoint test loss:", test_metrics["loss_total"])
display(test_report)

for calibration_key in CALIBRATION_KEYS:
    subset = test_report[test_report["calibration"] == calibration_key]
    x = np.arange(len(subset))
    plt.figure(figsize=(11, 4)); plt.plot(x, subset["MAE"].to_numpy(), marker="o", label="MAE"); plt.plot(x, subset["RMSE"].to_numpy(), marker="o", label="RMSE"); plt.xticks(x, subset["component"].tolist()); plt.ylim(bottom=0.0); plt.ylabel("observability error"); plt.title(f"{calibration_key}: held-out test error by coordinate"); plt.grid(True); plt.legend(); plt.show()


## 18. Compare with a constant train-mean baseline


In [ ]:
baseline_rows = []
for calibration_index, calibration_key in enumerate(CALIBRATION_KEYS):
    train_mean = train_target_cache.values[:, calibration_index].mean(dim=0)
    test_values = test_target_cache.values[:, calibration_index]
    error = test_values - train_mean.unsqueeze(0)
    baseline_mae = torch.abs(error).mean(dim=0)
    baseline_rmse = torch.sqrt(error.square().mean(dim=0))

    for component_index, component_name in enumerate(CALIBRATION_COMPONENT_NAMES):
        baseline_rows.append({"calibration": calibration_key, "component": component_name, "model_MAE": test_metrics[f"{calibration_key}/{component_name}_mae"], "constant_train_mean_MAE": float(baseline_mae[component_index]), "model_RMSE": test_metrics[f"{calibration_key}/{component_name}_rmse"], "constant_train_mean_RMSE": float(baseline_rmse[component_index])})

display(pd.DataFrame(baseline_rows))

print("Aggregate model metrics:")
for key in CALIBRATION_KEYS:
    print(f"{key}: MAE={test_metrics[f'{key}/mae']:.6f}, RMSE={test_metrics[f'{key}/rmse']:.6f}, MSE={test_metrics[f'{key}/mse']:.6f}")


## First ablation if training is noisy

Set

```python
USE_SAMPLING_RATE_AUGMENTATION = False
```

and rerun from the augmentation cell onward. Sampling-rate reduction changes the amount of information in a window directly, whereas the cached target is computed from the original source sampling.
